# 🚀 TASK 6.6: FINAL LOCKED BENCHMARK TRÊN GOOGLE COLAB PRO (GPU T4 / A100)
### ĐO ĐẠC 18 TẬP TEST BENCHMARK TRÊN GPU VỚI DỮ LIỆU GOOGLE DRIVE 5TB

> **Người thực hiện**: Thành viên C *(Training & Evaluation Lead)*  
> **Mục tiêu**: Nạp checkpoint `fatformer_crossdomain_weights_only.pth` (2.31 GB) cùng cặp tham số hiệu chuẩn đã khóa $(T=1.0811, T^*=0.3700)$, giải nén 18 tập test từ Google Drive 5TB (`test_benchmark_gans.tar` + `test_benchmark_diffusion.tar`) sang NVMe SSD Colab, thực thi kiểm chuẩn đo đạc 1 lượt duy nhất trên GPU và tính khoảng tin cậy 95% Bootstrap ($B=1.000$).  
> **Môi trường**: Google Colab GPU (Tesla T4 / L4 / A100 SXM4) + Google Drive 5TB.

## 🛠️ BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN
> Mount Google Drive, tự động phát hiện thư mục `Fatformer` hoặc `FatFormer_Hub`, clone/sync mã nguồn repo và bung công cụ đánh giá.

In [ ]:
# ==============================================================================
# BƯỚC 1: KẾT NỐI GOOGLE DRIVE 5TB & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA
# ==============================================================================
from google.colab import drive
import os, sys, glob, shutil, zipfile, base64, zlib

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')

# Tự động phát hiện thư mục gốc Drive
drive_candidates = [
    "/content/drive/MyDrive/Fatformer",
    "/content/drive/MyDrive/FatFormer_Hub",
    "/content/drive/MyDrive"
]
DRIVE_ROOT = None
for dc in drive_candidates:
    if os.path.exists(os.path.join(dc, "datasets")):
        DRIVE_ROOT = dc
        break
    if os.path.exists(dc) and ("Fatformer" in dc or "FatFormer_Hub" in dc):
        DRIVE_ROOT = dc
        break

if not DRIVE_ROOT:
    DRIVE_ROOT = drive_candidates[0]
print(f"[✓] Đã chọn thư mục làm việc Drive: {DRIVE_ROOT}")

# Đồng bộ repo
REPO_DIR = "/content/fatformer-xla"
if not os.path.exists(REPO_DIR):
    print("[*] Đang clone repository từ GitHub...")
    !git clone https://github.com/LeeVHoangtk3/fatformer-xla.git {REPO_DIR}
else:
    print("[*] Thư mục repo đã tồn tại tại /content/fatformer-xla")

# Bung công cụ đánh giá mới nhất tự động (Offline Self-Contained)
os.makedirs(os.path.join(REPO_DIR, "tools"), exist_ok=True)

eval_tool_path = os.path.join(REPO_DIR, "tools", "eval_locked_benchmark.py")
b64_eval_data = "eNrVPF1z20hy7/wVE2w5B3gpiJQlS8sKt0qWLNtnWXIkeutSDAsFAUMSSxLA4kMfp9NL/sLlIQ95uqf8htvH/BLnl6R7vgcEKW9yVam4yiQw093T09PT3dPT1Hdk5+UOibI4SWcDUlfTnSNs6TiO0zkLq7OsWNFi5w/nxwPyMaErcjKvw5R8nGchuc7wLZ2RkzqLyN+T0yyBhoTW5H1Yk4vZHOHdUVguyGv/tdfZIRdhTioK/4sM0MqMFHVFZlk6INOwmrKhgqjIyjLOVmGSBnc0mc2rMsjS5YOfV3MgcYqYNyF8REhsHq6QDhs1YqyJ8fYHZESGpO/3jvr9Lhm9hJee/+qw1wMioySdkyoD6AVMBGhVSQr0HsgPBy/ImyyryqoA6u6bIeD3eh6QzqA/gu8Eh4uyiiyzBCi9z3AyZGenomUVxPVq9QBAQGmBk6/mNcDNEkCsI3KTAcIf6rDi/K9CQEUWaEV+qUNycv0TuQ3J768vLwAAx8vYKnSSVZ4VFclK+VQ+qMcqWVH5/HOZpfJ5tsxu5HNYzPKwKGlnWmTA00NOSyK6ruF7SS/CFS3zMNIQoAwS5DSJqi45T0r4HNUA3SXH6UOXXOZVkqXhUvGX1qv8gYQlSXPFXVZEc+vFT1MGkjZb/WmdRpwiApwJVlhvXSXL0o/DKlRMwXNJgSN8OM/CmBYGwm1SAiEJi0IuUblKDvL5w7ns+rAKZ7TT+Xx1+fu3J6Pg6vISdSYr/Tys5n54U+K3K99/zpJUvcRJkYLY3CCYJksaBF6XOL7veF4nmRKLYArKAuoFi8YwBx0C/+Sbn6QlLSq317WQvA6SmYdlWFWFi8BlFWc1TNgpaJSl02RWF9TxNDHe7xu9Lk35th46bFs7XgtNWhRbaUL/MzQ7744vgusvb67fjq5BeGOG7eRFNgtTB2iX1cOSNp738OUmmbFmjhA9RAYYqCx/nIW1eIopzafhgjqdSadz+uHs7Mv1h8uWkWd1EtMYMZbxKtjr9YzHIJrO5GsfewTKElCCg16wd8jGZK/QD//td+znKHG4XFLszNN4hd+3v8TJdFqj4jEOj8/Pg/6RwZ4ppu/J2gQ6nU5Mp2SapDFoVAHGhN7DngNRu1GYxgloPy0HbB+O5dYbg5maTDyy8yOxmvgKgsqTiDDDpvBZB/4DTUC9jHQL/oN1hhFrqhqjADZBy5aIPJOS7GUcly5D8mzKBa3qIuX0VMcqrKI5Rfpornz8cKMuwEZ1USa3dDgqamqNJDBaaTd5FLDj3oSTEGAXWQpbvhMtw7Ikp2iw34TR4gZaXbBCn7K4XlLBPFhf2UdmSUiW4BPQEzRsO/gQ5o5S8u7zl12J4aPtRiq4rEGQpEkVBG5Jl9MumsqAMUDL4Z4hqbLOaeF6vgLXk0dEH9bnFqQFfJ7A017svuqS/usugS9Yd9DS4V6X5GHMdmi/gTyNOOp5ktKwcPuvyUvS7+/xT4sn2NSScVCiu7CIBd/3Bq9zoHYGtmFZu4o3997zLIC5f5vQO3ful8kfqdsDI7ljsCVWRHDnzj21Lswwo64LQ++Kb70wooHEEHrEIbjzMozmBB9gYSKIMgh49kgu27bFSCq6kjsLnR1uIebjJhPoLLlTH4D3zpY45XBZUnPJkHlGAjrZd6NLEMBe8WgDKPcEENpVwQKDiyqpO7ZU3QC4okym7t7efpfAh+d1N4GOshFNyww0ayPIBXyGSyS4omE6HPf8/aP+/sH+a9AueD44PNo7PGCPvaO9fu/w1cQm1foP/AdS2nt99Hrvh4N9RN973X/V2zs4Ys+HB4eHh/3+xGBrYuheECxpKpbJkLjQGuhztfAttBmtsFGvcGwqLnaBrDXuGPonqnsZ3tClWMuxw96ciWmDrGW1LdF3EFChBjJpS92zIO5xlVlQA7KPU9zAagEVIAUVGzRWqjES42U1A2psq/hZDuLgLLN4BC2gM/HYrsT4wrl698bx1kjcS0EoVXCBqg1H7yOaV+Qt+wInM2ilwif1RwrRe/ukxLLdd7mEhb+LQM3rigY3SRoWD8GKghmLyuAuqeZBBAp54zICaGgr2KR8FK7NXGkwVKAbuugKzGkIw1I4YCwzCLzVgYADzAtazrNlrLvFEYH5VG0MWO9E2R72/V4fOVI4fqAlghMHBPD5HJoAIQH6NCPprEb3MHrpW+glzI7GAZ8YDCwedk2uGSBEUswFn/llNq1W4b1roXZJnKyG0qpigBRIDPY9HoCLmPhRXoNfYSG6cCpCcEGa4+rxlwaYGJ/GSM01aP841LLz/LDEM4WbpJVAqfKgoCFuojT3y3rluoIIyNeDg6Jrjo1tnKO0gdWEEl48XAZhhL7MlePsEhQLR4fZeh76tF7P7yl2kPc2dvot7PQVOzZWE0oLXLPDMCQ7+NLCziqEI67A6fkH0OmqSX2vCHLyGWxeiDTlCIITxb5myROjolU0WluG/45cXZ7sHH85QV09/iymCr4GZ8p4Fk0pnbEmZK4jY0YOCIwTMHACBt60TQhriOBC1KkDGLLLPjvrRi0r4GjBZQvcwpatDP0yjEaYLjgU7IrqIVgmC+oyXBtmzNomkiAcrykThUFTI8AUAkTCGTPkxspqa1/r1VdIYBqEGGDh+MMO6tEu2RMzFWLABauh2ZXQTFzN5eBeq4yCFonstIkEe2D3C56JqQRjTUjPocpLTjWqVziRJr5SZabOzwL3DDlCoM5O2oyLHI2Xi1/fIxkNxpTBUF2JA+Jr48VT+wek1qK/wo88KvqO2lCOsOOuajEiC8fYSwrQaDNB5X5UcLLBBJI7VQGprWsAgR6ofhTrMsldtkXY5mDT8iz4fB08b4N+ajpQmbUKouT/xGd2hdlQjAzg7Mmwe7KzpDSWrft7TSfLkktjRlR63IbLZWm7ZzN2LQk72/WCYDH4ylY+MuTih7T4ZYj5MOaPtSH1tPWT/f9ADtYC0scnrqA3qAOgufBdi2+2q8ZwnOD/O/+/IgBMJASYSOB21VhiI6rGtgACaW4+hHxhFZIIMYTc4JgKR4yh8V7QfAmJx8ZJv6xv1Bz0hMZyjIkF2GoJNahprnggo6lvjmO09XwmlLE40LFMWzyzBqo1SDj4TQFNx+RneyzTGKRv8WOjroFqN6D5aY1oND+rRiAjoxhmB5XhVkrCtoYf5nBaid0Vj3OMTSN4Q9/HGzhLUQKqfoe6wAwj7xrDOrlAfO+AuVatkxOFNIfbg3asHw5bsJ5zL2BZgx8OwD67RVYD/5wtOOZAmKVacExs8iwrfVMnS9yv0QK27SqL6dKF3FS0yCGlXLHD2oAwI4gW33yP6S3sIGmp+dszWQkwc3A9A6cPuENY4dUIGM1bfJeXLnEo711I84JF5UnA2q0fcfMCpeeMX06IGsHKoHGzO6d4FpJJMh+z4k1b2ci7Wekwv8pcPlFpfzBrXxaRzwSn7i64TFkbj6vxKcDYCQTSuNhwtQqKQYfOyfmHz4OfktHO+W5/39E+WC3BUD3pTotV3VyXNCiLFTNjdussxBxuowNAA8jWB2i4+z04KrMer6OFPGVSPkVPt0BRx3jPBiJNYRkhtLRu5oh7ffWJvNr5SIsU5fM9eccG9bTomWxALIbMXC0vIWdY84ZSQj4tXsvu2iBeUz0MzlOWL5XgzHOD/g3IY4PGk6Ega5kOuAyoqEowwFaOmyx0wTrluLlCTE8MHXBhkJE3VXvI90enkdIYgbl/WxRZ8bcY0cpTQ4ibApkUNI9RQx8dVR6Tp8MFz9oddjOET+0ssO+xhWE4PwH06C7Gh4MJjrvw8e6kKjF/4iIaJLNBB9jhiyy8Abll3nzRhQc5Mk+Cud6TsUlxC7FB2fT5yAFOQU4H081R1ZTrKilh4rMgAmtYiSAK3Au7cCrQ3TsSYkEfSgdjIcM/1Sm9z2mEJ4LNBAygNhpKB//r3/884ffNc0wKQx4Mrrc/8dGHjxajT13yRVEdPjbZkNppH2DVQH83ISc4AlzZDtAsstgULV/4YCp/BfmgNb334b4cwCzrPGebPExw61p7l6+HYRxVp09vw6Vr3XJws8idD16lu2KfsotgPGbKS2H/uJjVK5pWn1mPi0fIIsm5Xttm5izBy9lz5sTIG7gJnK/CYmHc8AtB8TF8uIZA68KIu87Ojp48LBpGWkPh4aZhvayGzunx6BhuwnY1XLn7bWUBz4wrrXj7sMwD7PT3gdB2OkZYLimJw4qkJYsN5nSZD50LnRWMVdGCrE54ZigZlG4YSJy45EA8wwjGCMscfvNYWLhQQBzULh5HjjKC+64VuJ5ZFrG0J+kfgZpiPUf5jODA0+XM2RXbFx6OM+V2Sjd4pRfgGUJSAi0x5P96Kzb67rusWNCibEXvbcfWwWL74HjufFbQLKIC/DDiO6wEJ0MD8OlUCfpEWA5UHX51ZN83bhcRNw7tgo7qOHTQTXDPhq94kRHehgnk5JfUFY5C+DM2jIinxGjsi8UMOj3MYsKhA/H00ZEZwjhnx6Ozy6tPb694+dDZh4vjc3J+efLx7Sl58/bi5P2n46uPcGb5fHk92nl/eUJ+env14ezDyfEI7sXBrBxff7TMirC4o/WaH4hoMbiAkh+9R1n5zyOy6ptJDX9/+kT+REZyh/HKIAGnchsItTYsbuebZCCAuaCR1glfK3lD7LLg1huoweWy2xS1yMTNFZJTUQd/dY2RPJm+PWUBfWI6F8jkopUTDqDMlrfovBY5utC2igJ9qcgG0JSMTNT/whw/Q2W1v2ui8+DmBsW02m/i7y7C2WxJd5MUTMjuy5e/jQV18y9FAkL6RpGsh/6WpzCa4QAORgEu1ePdDRBSCFshJw1333JktJa2a09LHhX14XDYUD+lQH1fVPHdJMbVOWbLpD0XUosglVwGNw8BZArwrn2IOS4R4TaIs+uArDKa0aXI1sYRwgbyth8wYzB7uNNXUFNouBy4ZgNndFtr4wi7sMLwyz5wYrgL7GO0a5fkNGtIGpMdw9dEVRWZ/x6NG1ZIgjps/sEjwD8Fj8mTn6e8xojdHEO6k7wge09rVJCvRCfV9nv2peukJeT8jvxjTSstAVno4M55EWJMoY7wARQIBYdhKPfY3uYD2i+MHkSrpmBv2CGTxcuP9ko9/Y9Ei2km8P1GFRErpLNJdxHMFoEoUZJISQlEXEHMW7+E/g7SwyuuDUIQPKS2x1lD0zVI49aChpxNM8dJ6iKlrRNxXr50+HQ2lzI15mnPMV+Hm3RasFqLodalruugOk2V52npzlZuNkscBcO0PGBhxvJmiVIau06PJS5BCFjw4zp9ljd0MHGIr1af1dPTKKxt0j61qE2dBJtdzdE3SjvaMDtzlnxhA7iWSDD7n7Bis7twuXgWW1KYpojF0AfPFs4Aj9MUTt53rBgMUqX8PO860rb4P+fym/KHO3qTQ/Hp87TlysskrG3NLJHyWU9Tz7BmsMhP3rdZThzFSFxnFVyhTTFNinkNSD/j2f7WY9JhyYg1Qj6camsKkaZpcgBfdDPlXVj5jHUSIreBIhXj/Uh6kw3pgpE8t1fsPIUfjwbfT7y8rAD7+Mgul01mvKddw4iC6bR6n+z0kCUM6/rc9ILbMgvYnoaZsHHC0rOhmXN/AB5LViRvniOaLtM257/BpP9Gj/m385rf4Dm5uRShzp5PfkIBoQggvinBKyxrUfYPZwgzHP/nFN3iGzhExxBpKEgWIGFoz9bdWGItPIjD4Hhl2FK8UeZ3c1CwUFZ2h7yk0j2db5Q8mjhZabiu6UAUjSDLhzXLfBXmhnJfXYkgqiqH6xWYisa2CFMV0bGSfCCj6/PdWNbs61M8J6DfwW3O6+l0SXlikReliiM7hzUajFso5E3do443XQ+ODQdY9fCoBT+e8PHDqLNFSyt/lJAFswKSvw2riisFVWwPuFR8nu0VcffNXJ35zxgGrkL9sK4gmVxWAt7Hw3uX0BRP5fHQbEV7wQ/ym6w9pFtkthDqcTttgQGXlXQB+LsBfu3Lb8q8diR5QyywHgz5Uzj3ljS2RQplMZW++oPslF4ikX8IuVaZ19jGZkYM427XQhHtCkcf3uytJ5nVo3sWqLkZTVh1869T27wyETj5xpLF9Yl3zSk1KnF1hmLYTFk0IGWOYminLBSQnuAMLiXzAH+lAlw78PuDkiV/hJUxf4/Asz6n+gcMxk0ARLV49fpoceEw2mCn9Rg2mw63SgCBNsnuwj0sCgAwogBPaoq8AWwW9whRj43GRhmyWbmjoFVbE9io4FHAqq0JzCt5FBy+roHkFkS+BiD2SVBSpMXvjEUb3hkrYOsqBn2L1E2xHutHOoK1NONHvRyDH8qnCUH/OugflJim+oRCI8cnJxCcSCZ/pyT5u8nA35s+vQBAp7H9p84VSNDEkhI1kM5AbiaIlONWulCFaOKATA3w489WX6663EchM/Zeeo7KbLzyjZ/2Gb/XY/HcPIPc3CcaMtWHzcgelc6L90teEMYFDL88YuJmzqPgJwEeZnKHD5upGIutMGGGmW0y7n/w50C/DVnvv4muZgeLEgVYk+8ira7YcCxMYodM2GT8zVv7cQ8itP5Shv0Eplm9tG1/8+E2b3DFxNadjkE/TNlsnBhyAW63bX5VFMeEMS4sK2CTmTR/C7Fe0GfS0fbhOTJrJX8mGW05niPTKApUFJhNeRY5b8fNn0dtMz9iUcyutUWxTRPPUsK4Ae4PdIhKRdWG6YqtAN+f3oKjgZdrWYqAmLg5LEy1W7rmPpDo6tdyJg1Ru2mREZsL8MRGViQu4XIBAmttK/b9b6gnxHoWZkxuMkxAWjXZGyKaRgRihTUmalto0wxJLGRW0hYlZgSyVvPZzl+3bWCtG98WfGwKPIzaGM2QCNZ1g1HgAkVS7OSGOW4+JXZ2WauzgjwQK3tlRa8d+6jmQJFL653Tm69//cvFOzL6+uu/wtf7r7/+5TP5+OHrr//yiZy8//L1r/9xQT6+/88/H5NrgMAmgDr58vXXfzvZdOFkD4NpN6n67ZUvqG3Mw/Cugfa7GEUp5HbXi2X5NlTTJza7lVt0bAZxQ21hUO2xVgYR+XkGOdRGBkX3JgaFSm7hUWzZDUIU+G1suriJTz4MICCSygbZT97ZJbqtL9rgJ7QbopP10TZOV0OYMxbG5sDnP/zHHEjrD/3Zxi39FfgQSCiKKxJ9cQ5nQbw6CbKFkUjGCrKovG1NrJuokHmBwg0sp4f7kQhu2kTlRgCZ4SK594GGuu0FGwHVL1BeJ0MXNEPSrG7bAg1MVeMpIbdp5zZchHxGcbahZzKeY9Bzisd1FpGJCKerApmuHa50zeija0QQXSMM6HJf3mVeuWs7WB4FsnM++3GeWC2AunPwaN/4HT3+0QNDJFP/roBsi+t0Hb6qgnUPDR/YP/sWhvnqhhgaSYsmvRIrqsbzCU/AzpFA6xByV0J6jOVIT0Nyj5qs/noF5hL5LbxIqkFeDgJ3Md0nR9XVMs7wb1R8q8KKu0+tr5yGjzQEd/gIOcUHzMNYh1QHDg4hZpwg0GmEtpCFWkCrI8uWNtQ1OY3gCZMaUPy2wvCLZTlAglOW6XBe/NPOi9XOi5i8eD948WnwAkKdBrJR+6ST7DeQDmN/Q8K612W5avsSnx/PL3aP13jSjpqnHuAZ/vbBgGzPITjKebdhrTt2hqODDazZTyL8ewISx3D0myJ48Ojg1/G8jwZFmuCmnGRmkV1463nIBKOOQo3bdYhgYb0e1hda2R52KhfP3RYY/ZcbBto8tQEKayLBxOsGrljiX6aHMN7mRrSjD/jaOBi74xsNBIL6IJTcNfcAXNdAejaNoRgI/xQA/LAHVSMsoyQxazXtaw+5pYUv4o5JHZnZbjb421jFgn9dJGC5hyBg59kgYLUZgTMQv3vEMsTOfwNzMtLe"
with open(eval_tool_path, "w", encoding="utf-8") as f:
    f.write(zlib.decompress(base64.b64decode(b64_eval_data.encode("ascii"))).decode("utf-8"))
print(f"[✓] Đã nạp script kiểm chuẩn: {eval_tool_path} ({os.path.getsize(eval_tool_path):,} bytes)")

plot_tool_path = os.path.join(REPO_DIR, "tools", "plot_three_way_benchmark.py")
b64_plot_data = "eNqNWMtu48gV3esrLtgwQE5LHJJ6tCRAC1vuTidpP2JrJgEcgSiRJZExVUWQRbc0g1n1YpbJYJBFkCwyCbJIgCyCLALYiyw8P6J8SW4VH6JkudG0JJJV91W3bp1T5RfQ+qwFHvdDthhCJuatvmxpaJrWeEPEG54sadL61bvjIXxJ4SSkGZxy/IQwDuRLG04oA30SJJS2fknWMObLmCREhHdUdnnBkiS3BkqQ9BZ6Zs9oXHNICQtAJJkHfkjZsGGbcEJSGoWMwk+4B+MvL6/AsZwO6Ge20XBMOOMQhKj0BkVEhmKXEWFwXNjtmjZ8DmeO0WhvRU8JnKF1mHCUPJVP+lkY0VRwVO9J+Q68IVE0I96toQbcCJcxTwTwtHxK19Ujy5bxGkgKLC6blkTEERdRODPjtXyS3XEkGo3Lq4ufvR5P3KuLiwmM0KAZExGYZJbKu16+/4aHrHrxw4SRJdVdd45huq7RBM00NcNoXHwxccdvj68m7uXx5G3NntKv+0KV0+PJ8fXriYaPEV+k8i4wRy7m3hVymtz3ZO16aprClDMzZgvNwLHPISApESLRcdRmKnyeCVROqMfZPFxkCdWMYQPw2vabtV4dZ1vV0UhTdSSNNnw6xzRhlIUqZgdV1xE1s5TqWkrJjCesdWe5/db7IBR0kYS+BhjM832sZoXckTAis4gCjVIKGvojWSTQt/SGgTVBJyu7CWTlGJg5pZnN5GSlOjY7TSmUhl/RkW73mlihXcy7H4ejtmXhAKSVF4D1eZ0rgT3EUk2w+oksj6KUu+AFoXzzUCLioVLzCEbMk5Cm6PdGNclLG0cUC/KMeAmH4/H410yfbB5+zxYggs3DB0Nr7ou+Ibe0kDwPNvf/ZLhqNg/fyqKez7M05OyA0hUlUaF0srn/KwfBH//MYLl5+C0wyg9pXIxbx19IhSs5pgX6+GMIcfD4Fwbi8Qcv2NH58Xebhz/ABCdjjCGt4fX4NWqO0YcayP3fYvDyl83Dd6JUneYZXdpu6vEkz0zXMi2rCfmvbeX3rm32cTo6fdN+NVVzgA4g9UhEfVihFPgUEGK8IFMOCf76HI5y807NfH9gOmip75gOTvagZ7bR/KBr9nH2u2a3N81VOjWVQS6Lmm0UGnTMQU9p9rtYMOZgUIxihbIsNhHt2ILqEWX6dsqNvADfh74IUMwynW6uhAtGpDY2YVWaM5LoK2jlYs1tWpplC5Y2jUbaFg8rmJTAiGvb4xFPRtqLvif/sIH6C1o2dtVVrAXl2Kk7bm4Tte+wQNcKbfdgtua4bfedmb/n2Op37YFXd9zeGfHLasSdZwLY4R6J1PrL0zAVYRTByzFnaT0E2uvZlr0XwgzXr2Vp1RL+KcOKJkg6IX6pXEJY47hcc5zgSR6mBJebfIqaRcaKe3s6rIq/FJfSqnPbJS854bLZXFDhBjRcBEI3diRkJghjXGC96HPt62Bo2vNvjmrr68m1Wo/0yuhKNzCH1atKHTZ9LuEsMD5qRdCVGOm4BNofk5NSHueJn440Pp+nVECMXCPSj4UYkJHmUSZognNxhy8zLgRf4suco6rE2FdmN397r9IiRSK/nCSZFfTkilBEyA0SYb5T+4zN/X8zmCSbh395iHkK+nDf8fgPNtyuh7u03BLgE9bLz4PHfxO4RmSt+7edA+6bEBMfu4ydKFYi9G5TfXWgVdVoWlvtNQ+DZ0ZYN7JWBnQNcf8/8A7HA/qRsROm/Qk2wqWcRtvubjsiuqDM1yPuIQfHMU0gonNRtzzA5wR3GSSKcbosc1AtEGfLcc4QrohfsdxMbvTkRg1mGW7OWIDbPCYBN8q8wrWDG4Elv6N6GYsEGiRWk/i+W/BtQbf4iXlEktEkyWjhPZHeXMznx6myzj8V91X0+CyjPe04vqy3Kaq8RBCiyR1uXDnbUZDbGx+3NgiCdgtJaEtl8vccA5bIvx1BkQG2iHIyYbgs5xEnQj834DNw8IukEYcKRZiCEMUf58a0rvlyVDzdDO1pOUfnCIkkwnmEJUUk81JkdbBaSId5Gt2l/YRRCybtOorxSn7VbaQdy+z07VcyKmyeliacA6yZ0yVSoNOvOLQwYXW7vX0TnQMsmtOnvDkVp5YmnMGgMrEdCaZA3lUCqtjyRqfe2CkaO9tUyYpUNZfnsKkMIbkgUiiwHNlymcpXtY0cacNDXPqEfRXpGs96cHY99Hc8tFqHWHOXb3XJrc/b79TtO/v2DzFiYX4HD0GvTj01X3jmiPZ8PbFWgoaEykpRIXZABXFzotDz6kYuMg7I4CEHWQvXmN6yd/sLvC2LvmVPD/QXyLtdbZ+MvLmRde7kxsH67eC3h99+vu2cHpAs3N1ojnUkD1Kd/NbLb/38hrpH2nSbrp66sEdF1d8zW8K21d1LT056J0hvHzKQ3Pd9AcInhMMvsscfBJz/+AFR953iwWuSwVvJhRmidLa5/zv7VJ6rRfSULxIpj5KzGV+5grsET/A4Kt022xi0aRufwCXyjCWkHTciazwkFqSAh9YlQjVWQPrkzLt7wkUndIW7PZffFixRHR3JHUVa2ZOvDmxF3CEGTXHnooLQttpexNOSouIEtzO497r535++n8r/FawyUlIdx08oj3TqHxwzKiF6CF/vOv1GnW/xoOq6cgiuC6MRaK4rT7uuq+Vbwvzo2/g/FeciVw=="
with open(plot_tool_path, "w", encoding="utf-8") as f:
    f.write(zlib.decompress(base64.b64decode(b64_plot_data.encode("ascii"))).decode("utf-8"))
print(f"[✓] Đã nạp script vẽ biểu đồ: {plot_tool_path} ({os.path.getsize(plot_tool_path):,} bytes)")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài đặt các thư viện cần thiết
!pip install -q ftfy regex tqdm tabulate matplotlib seaborn pandas
print("[✓] Hoàn tất Bước 1: Môi trường sẵn sàng trên Colab!")


## 🔍 BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ CHECKPOINT M4, CLIP BACKBONE
> Xác thực GPU phần cứng, nạp checkpoint `fatformer_crossdomain_weights_only.pth` và `ViT-L-14.pt`.

In [ ]:
# ==============================================================================
# BƯỚC 2: KIỂM TRA GPU & ĐỊNH VỊ CHECKPOINT VÀ CLIP
# ==============================================================================
import torch

assert torch.cuda.is_available(), 'LỖI: Vui lòng bật GPU trong Runtime -> Change runtime type -> T4/L4/A100 GPU!'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'[✓] GPU khả dụng: {gpu_name} ({gpu_vram:.2f} GB VRAM) - Sẵn sàng cho Locked Benchmark!')

def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

# 1. Tìm checkpoint M4
ckpt_path = find_file([
    f'{DRIVE_ROOT}/**/fatformer_crossdomain_weights_only.pth',
    f'{DRIVE_ROOT}/**/crossdomain_model_best_m4.pth',
    '/content/fatformer_crossdomain_weights_only.pth',
    '/content/drive/MyDrive/**/fatformer_crossdomain_weights_only.pth'
])

if not ckpt_path:
    print('[!] CẢNH BÁO: Chưa tìm thấy fatformer_crossdomain_weights_only.pth trên Google Drive!')
    print('    Hãy tải file fatformer_crossdomain_weights_only.pth lên Drive hoặc kéo thả vào thư mục /content/ trên Colab.')
else:
    print(f'[1] Checkpoint Khóa Sổ: {ckpt_path} ({os.path.getsize(ckpt_path)/(1024**3):.2f} GB)')

# 2. Tìm CLIP ViT-L/14 Backbone
clip_path = find_file([
    f'{DRIVE_ROOT}/**/ViT-L-14.pt',
    '/content/**/ViT-L-14.pt',
    '/content/drive/MyDrive/**/ViT-L-14.pt'
])
print(f'[2] CLIP ViT-L/14 Backbone: {clip_path}')


## ⚡ BƯỚC 3: GIẢI NÉN 18 TẬP TEST TỪ GOOGLE DRIVE SANG SSD NVME COLAB
> Trích xuất dữ liệu chống nghẽn I/O từ Drive sang `/content/dataset_local/test_clean` trên máy ảo Colab.

In [ ]:
# ==============================================================================
# BƯỚC 3: TRÍCH XUẤT 18 TẬP TEST TỪ GOOGLE DRIVE SANG SSD NVME MÁY ẢO COLAB
# ==============================================================================
import tarfile

LOCAL_TEST_DIR = "/content/dataset_local/test_clean"
os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

# 1. Tìm các tệp .tar trên Google Drive
tar_gans = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_gans.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_gans.tar',
    '/content/drive/MyDrive/**/test_benchmark_gans.tar'
])

tar_diff = find_file([
    f'{DRIVE_ROOT}/datasets/test_benchmark_diffusion.tar',
    f'{DRIVE_ROOT}/**/test_benchmark_diffusion.tar',
    '/content/drive/MyDrive/**/test_benchmark_diffusion.tar'
])

print(f"[*] Tệp GANs tar: {tar_gans}")
print(f"[*] Tệp Diffusion tar: {tar_diff}")

# 2. Giải nén sang SSD NVMe máy ảo Colab
def extract_tar_safely(tar_path, target_dir):
    if not tar_path or not os.path.exists(tar_path):
        print(f"[!] Bỏ qua: Không tìm thấy {tar_path}")
        return
    print(f"[*] Đang sao chép và giải nén {os.path.basename(tar_path)} sang SSD NVMe ({target_dir})...")
    with tarfile.open(tar_path, 'r') as tf:
        tf.extractall(target_dir)
    print(f"[✓] Đã giải nén hoàn tất: {os.path.basename(tar_path)}")

if tar_gans:
    extract_tar_safely(tar_gans, LOCAL_TEST_DIR)
if tar_diff:
    extract_tar_safely(tar_diff, LOCAL_TEST_DIR)

# 3. Kiểm tra số lượng tập test đã sẵn sàng
subsets_found = []
for item in os.listdir(LOCAL_TEST_DIR):
    item_path = os.path.join(LOCAL_TEST_DIR, item)
    if os.path.isdir(item_path):
        subsets_found.append(item)

print(f"\n[✓] Tổng cộng có {len(subsets_found)} thư mục tập test trên SSD Colab: {subsets_found}")


## 🚀 BƯỚC 4: THỰC THI KIỂM CHUẨN KHÓA SỔ TRÊN GPU & BOOTSTRAP 95% CI
> Cố định $T=1.0811, T^*=0.3700$, chạy suy luận trên toàn bộ ảnh test với GPU và tính toán khoảng tin cậy 95% Bootstrap ($B=1.000$).

In [ ]:
# ==============================================================================
# BƯỚC 4: CHẠY KIỂM CHUẨN KHÓA SỔ TRÊN GPU (LOCKED BENCHMARK)
# ==============================================================================
import subprocess

OUTPUT_LOGS = "/content/output_task_6.6"
os.makedirs(OUTPUT_LOGS, exist_ok=True)

cmd = [
    sys.executable, "-u", f"{REPO_DIR}/tools/eval_locked_benchmark.py",
    "--checkpoint", ckpt_path or "",
    "--clip_path", clip_path or "ViT-L-14.pt",
    "--temperature", "1.0811",
    "--threshold", "0.3700",
    "--output_dir", OUTPUT_LOGS,
    "--batch_size", "32",
    "--num_workers", "2",
    "--n_bootstrap", "1000",
    "--device", "cuda"
]

if os.path.exists(LOCAL_TEST_DIR) and len(os.listdir(LOCAL_TEST_DIR)) > 0:
    cmd.extend(["--test_root", LOCAL_TEST_DIR])
else:
    print("[!] CẢNH BÁO: Không có ảnh test thực tế -> Chạy chế độ Dummy")
    cmd.append("--test_dummy")

print("=" * 80)
print(f"[*] Lệnh thực thi: {' '.join(cmd)}")
print("=" * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError(f"Tiến trình kết thúc với mã lỗi: {process.returncode}")
else:
    print("\n[✓] HOÀN TẤT VÒNG ĐO ĐẠC KIỂM CHUẨN CHUNG CUỘC!")


## 📊 BƯỚC 5: TỔNG HỢP KẾT QUẢ, VẼ BIỂU ĐỒ 3 BÊN & LƯU VỀ GOOGLE DRIVE 5TB
> Hiển thị bảng ma trận kết quả, tự động vẽ biểu đồ đối chiếu 3 bên (CVPR 2024 vs Plan A vs M4) và tự động đồng bộ sang Google Drive `logs/task_6.6/`.

In [ ]:
# ==============================================================================
# BƯỚC 5: BẢNG TỔNG HỢP ĐỐI CHIẾU 3 BÊN VÀ ĐỒNG BỘ GOOGLE DRIVE
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

res_csv = os.path.join(OUTPUT_LOGS, 'final_posthoc_benchmark_matrix.csv')
res_json = os.path.join(OUTPUT_LOGS, 'final_locked_benchmark_report.json')

if os.path.exists(res_csv):
    df = pd.read_csv(res_csv)
    display(Markdown('### 📊 KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 18 TẬP TEST (TASK 6.6):'))
    display(df)

# Vẽ biểu đồ 3 bên nếu có script
plot_script = f'{REPO_DIR}/tools/plot_three_way_benchmark.py'
if os.path.exists(plot_script):
    try:
        subprocess.run([sys.executable, plot_script], check=True)
        chart_p = f'{REPO_DIR}/DATASET/logs/task_6.6_three_way_comparison.png'
        if os.path.exists(chart_p):
            shutil.copy2(chart_p, os.path.join(OUTPUT_LOGS, 'task_6.6_three_way_comparison.png'))
            print('[✓] Đã tạo biểu đồ đối chiếu 3 bên: task_6.6_three_way_comparison.png')
    except Exception as e:
        print(f'[!] Cảnh báo vẽ biểu đồ: {e}')

# 1. Đóng gói toàn bộ kết quả vào tệp zip
export_zip = '/content/task_6.6_final_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

# 2. Tự động lưu trữ trực tiếp vào Google Drive 5TB
drive_out_dir = os.path.join(DRIVE_ROOT, "logs", "task_6.6")
os.makedirs(drive_out_dir, exist_ok=True)
for item in os.listdir(OUTPUT_LOGS):
    s_item = os.path.join(OUTPUT_LOGS, item)
    d_item = os.path.join(drive_out_dir, item)
    if os.path.isdir(s_item):
        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
    else:
        shutil.copy2(s_item, d_item)

shutil.copy2(export_zip, os.path.join(drive_out_dir, "task_6.6_final_results.zip"))
print(f"[✓] ĐÃ TỰ ĐỘNG LƯU TRỮ TOÀN BỘ KẾT QUẢ VÀO GOOGLE DRIVE: {drive_out_dir}")

# 3. Kích hoạt tải trực tiếp về máy tính nếu muốn
try:
    from google.colab import files
    files.download(export_zip)
    print("[✓] Đã kích hoạt tải tệp task_6.6_final_results.zip về máy!")
except Exception:
    pass

print('\n🎉 HOÀN TẤT TOÀN DIỆN VÒNG ĐO ĐẠC TASK 6.6 TRÊN GOOGLE COLAB PRO!')
